##### 카테고리

In [ ]:
# from pathlib import Path
# import zipfile
# import shutil

# base_dir = Path(r"E:\all_pass_카테고리_붙인_데이터")
# output_dir = base_dir / "category"
# output_dir.mkdir(exist_ok=True)

# for zip_path in base_dir.glob("category_*.zip"):
#     press = zip_path.stem.replace("category_", "")
#     target = f"category/outputs/corpus_3_categorized/{press}.json"

#     with zipfile.ZipFile(zip_path) as zf:
#         if target not in zf.namelist():
#             print(f"제외: {press}")
#             continue

#         with zf.open(target) as src, open(output_dir / f"{press}.json", "wb") as dst:
#             shutil.copyfileobj(src, dst)

#     print(f"완료: {press}.json")

##### 개체명 목록 

In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path(r'C:\work\ner')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / '.env')
from src.postgresql.query_reviews import get_connection

- review_method = '일괄처리', review_status = '완료'대상 중 언론사별 기사 추출

In [ ]:
query = """
WITH article_ids AS (
SELECT 
DISTINCT press, 
doc_id 
FROM public.ner_human_review 
WHERE TRIM(review_method) = %s
  AND TRIM(review_status) = %s
), 
ranked AS (
SELECT 
press, 
doc_id, 
ROW_NUMBER() OVER (PARTITION BY press ORDER BY doc_id) AS article_rank 
FROM article_ids
) 
SELECT 
press,
doc_id 
FROM ranked 
WHERE 1=1
--AND article_rank <= 10 
ORDER BY press, article_rank
"""

with get_connection() as connection:
    rows = connection.execute(query, ('일괄처리','완료')).fetchall()

document_ids = [(row['press'], row['doc_id']) if hasattr(row, 'keys') else (row[0], row[1]) for row in rows]
len(document_ids)

12988

- 원본 데이터 가져오기 (ner_documents)

In [ ]:
import pandas as pd

doc_id_values = [item[1] for item in document_ids]

query = """
SELECT
    split_part(doc_id, '.', 1) AS file_id,
    doc_id,
    paragraph_id,
    sentence_id,
    form,
    topic    
FROM public.ner_documents
WHERE 1=1
AND doc_id = ANY(%s)
AND topic = '사회'
ORDER BY doc_id, paragraph_id, sentence_id
"""

with get_connection() as connection:
    source_rows = connection.execute(query, (doc_id_values,)).fetchall()

In [ ]:
source_df = pd.DataFrame(source_rows, columns=['file_id', 'doc_id', 'paragraph_id', 'sentence_id', 'topic', 'form'])

# 언론사별 10건
# source_df_by_file = source_df.groupby('file_id', group_keys=False).head(10).reset_index(drop=True)
# source_df_by_file

# 언론사별 고유 기사 10건
selected_docs = (
    source_df[['file_id', 'doc_id']]
    .drop_duplicates()
    .groupby('file_id', group_keys=False)
    .head(10)
)

# 선정된 기사의 전체 문장 추출
source_df_by_file = source_df.merge(selected_docs, on=['file_id', 'doc_id'], how='inner').reset_index(drop=True)
source_df_by_file

,file_id,doc_id,paragraph_id,sentence_id,topic,form
0,NIRW2600000001,NIRW2600000001.10049,NIRW2600000001.10049.1,NIRW2600000001.10049.1.1,사회,"‘<서부지법: OG> 폭동’ 대응 실패 <마포경찰서: OG>장…<경찰청: OG>, ..."
1,NIRW2600000001,NIRW2600000001.10049,NIRW2600000001.10049.2,NIRW2600000001.10049.2.1,사회,"<경찰청: OG>, <마포서: OG> <고석길: PS> 서장 등 직권경고 방침"
2,NIRW2600000001,NIRW2600000001.10049,NIRW2600000001.10049.3,NIRW2600000001.10049.3.1,사회,<경찰청: OG>이 <서울서부지법: OG> 폭동 당시 대응에 실패한 <서울: LC>...
3,NIRW2600000001,NIRW2600000001.10049,NIRW2600000001.10049.4,NIRW2600000001.10049.4.1,사회,<경찰청: OG>은 <12일: DT> <마포서: OG> <고석길: PS> 서장과 경...
4,NIRW2600000001,NIRW2600000001.10049,NIRW2600000001.10049.4,NIRW2600000001.10049.4.2,사회,당사자들에게는 아직 내용이 전달되지 않은 상태다.
...,...,...,...,...,...,...
185,NZRW2600000003,NZRW2600000003.100358,NZRW2600000003.100358.5,NZRW2600000003.100358.5.1,사회,"<제주도: OG>는 시범사업 추진을 위한 근거 조례 정비와 관련 예산 확보, 운영기..."
186,NZRW2600000003,NZRW2600000003.100358,NZRW2600000003.100358.6,NZRW2600000003.100358.6.1,사회,<조상범: PS> <제주도 안전건강실: OG>장은 “사회보장제도 협의 완료는 <제주...
187,NZRW2600000003,NZRW2600000003.100358,NZRW2600000003.100358.7,NZRW2600000003.100358.7.1,사회,제주형 건강주치의 제도는 65세 이상 노인과 12세 이하 어린이를 대상으로 건강진단...
188,NZRW2600000003,NZRW2600000003.100358,NZRW2600000003.100358.8,NZRW2600000003.100358.8.1,사회,"환자에게 건강위험 평가, 만성질환 관리, 건강검진, 예방접종, 건강교육, 비대면 건..."


In [ ]:
from src.ner_review_app.modules.entity_catalog import ENTITY_SOURCE_CTE_SQL

# 선택한 일괄처리 기사에도 배정·참고 문장의 최신 완료·수정본을 함께 적용합니다.
review_query = ENTITY_SOURCE_CTE_SQL + """
SELECT doc_id, sentence_id, final_form AS modified_form FROM source
"""

with get_connection() as connection:
    review_rows = connection.execute(review_query, (doc_id_values,)).fetchall()

review_df = pd.DataFrame(review_rows, columns=['doc_id', 'sentence_id', 'modified_form'])
final_df = source_df_by_file.merge(
    review_df,
    on=['doc_id', 'sentence_id'],
    how='left',
)
final_df['final_form'] = final_df['modified_form'].where(
    final_df['modified_form'].notna(),
    final_df['form'],
)

- final_form 기준 개체명, 분류 컬럼 생성

In [ ]:
import re
import pandas as pd

final_df_ = final_df[['file_id', 'doc_id', 'paragraph_id', 'sentence_id', 'final_form']].copy()

entity_pattern = r'<\s*([^<>:]+?)\s*:\s*([^<>]+?)\s*>'

base_df = final_df_.reset_index(drop=True)
base_df['_row_id'] = base_df.index

entity_df = (
    base_df.set_index('_row_id')['final_form']
    .fillna('')
    .str.extractall(entity_pattern)
    .rename(columns={0: '개체명', 1: '분류'})
    .reset_index()
    .merge(base_df, on='_row_id', how='left')
    .drop(columns=['_row_id', 'match'])
)

entity_df = entity_df[
    ['file_id', 'doc_id', 'paragraph_id', 'sentence_id',
     'final_form', '개체명', '분류']
]

In [ ]:
# entity_df[entity_df['sentence_id'] == 'NIRW2600000001.10049.1.1']
# final_df[final_df['sentence_id'] == 'NIRW2600000001.10049.1.1']['final_form'][0]

- 컬럼명: 개체명, 분류, 출현빈도
- 출현빈도 집계 기준: 현재 테이블 기준으로 전체 데이터에서 [개체명, 분류]가 언급된 총 갯수
- 저장 파일명: 개체명 목록_빈도.xlsx

In [ ]:
from pathlib import Path
import pandas as pd
OUTPUT_DIR = Path(r"C:\work\ner\outputs\final")
press_map = {doc_id: press_id for press_id, doc_id in document_ids}

entity_df_ = entity_df.copy()
entity_df_['언론사ID'] = entity_df_['doc_id'].map(press_map)

# 1. 전체 개체명·분류별 출현빈도
entity_frequency_df = (
    entity_df_
    .groupby(['개체명', '분류'], as_index=False)
    .size()
    .rename(columns={'size': '출현빈도'})
    .sort_values(
        ['출현빈도', '개체명', '분류'],
        ascending=[False, True, True],
        kind='stable',
    )
) 

entity_frequency_df.to_excel(OUTPUT_DIR / '개체명 목록_빈도.xlsx', index=False)
entity_frequency_df.to_csv(OUTPUT_DIR / '개체명 목록_빈도.csv', index=False, encoding='utf-8-sig')

- 컬럼명: 언론사, 개체명, 분류, 파일ID, 문서ID, 문단ID, 문장ID, 출현빈도
- 출현빈도 집계 기준: 개체명, 분류가 문장에 언급된 총 갯수
- 저장 파일명: 개체명 목록_출현위치_(언론사ID).xlsx

In [ ]:
# 마침표로 구분된 식별자를 각 구간의 숫자값 기준으로 정렬
def _dotted_id_sort_key(value):
    parts = str(value).split('.')
    return tuple((0, int(part)) if part.isdigit() else (1, part) for part in parts)

# 2. 언론사별 개체명 출현 위치 및 출현빈도
location_columns = [
    '개체명',
    '분류',
    'file_id',
    'doc_id',
    'paragraph_id',
    'sentence_id',
]

for press_id, press_df in entity_df_.groupby('언론사ID'):
    location_df = (
        press_df
        .groupby(location_columns, as_index=False)
        .size()
        .sort_values(
            ['doc_id', 'paragraph_id', 'sentence_id'],
            key=lambda column: column.map(_dotted_id_sort_key),
            kind='stable',
        )
        .rename(columns={
            'doc_id': '문서ID',
            'file_id': '파일ID',
            'paragraph_id': '문단ID',
            'sentence_id': '문장ID',
            'size': '출현빈도',
        })
    )

    location_df.to_excel(OUTPUT_DIR / f'개체명 목록_출현위치_({press_id}).xlsx', index=False)
    location_df.to_csv(OUTPUT_DIR / f'개체명 목록_출현위치_({press_id}).csv', index=False, encoding='utf-8-sig')

- 2개 산출물 출현 빈도수 검증

In [ ]:
import pandas as pd

def read_excel_safe(path):
    return pd.read_excel(path, engine="calamine")

In [ ]:
from pathlib import Path
import pandas as pd

XLSX_OUTPUT_DIR = Path(r'C:\work\ner\outputs\20261124_final\개체명목록_기사190건_xlsx')
frequency_path = XLSX_OUTPUT_DIR / '개체명 목록_빈도.xlsx'
location_paths = sorted(XLSX_OUTPUT_DIR.glob('개체명 목록_출현위치_(*).xlsx'))

frequency_columns = ['개체명', '분류', '출현빈도']
location_columns = ['개체명', '분류', '파일ID', '문서ID', '문단ID', '문장ID', '출현빈도']

In [ ]:
frequency_df = read_excel_safe(frequency_path)
assert list(frequency_df.columns) == frequency_columns, (f'전체 빈도 XLSX 컬럼 불일치: {list(frequency_df.columns)}')

frequency_total = frequency_df['출현빈도'].sum()
location_total = 0

for location_path in location_paths:
    location_df = read_excel_safe(location_path)
    assert list(location_df.columns) == location_columns, (f'출현위치 XLSX 컬럼 불일치 ({location_path.name}): {list(location_df.columns)}')
    location_total += location_df['출현빈도'].sum()

assert location_total == frequency_total, (f'출현빈도 합계 불일치: 전체={frequency_total}, 언론사별={location_total}')

##### 전체 데이터 개체명 목록 테이블 생성
- 개체명 목록 테이블은 이곳에서 함께 관리 (ner_entity_mentions, ner_entity_build_progress, ner_entity_frequency, ner_entity_location)
- ner_human_review와 ner_human_review_add의 완료·수정 문장을 함께 반영합니다. 같은 문장은 최신 수정 시각, 데이터 버전 순으로 하나를 선택하며, 둘 다 같으면 추가 검수를 우선합니다.

In [69]:
# [실행 순서 1] 최초 1회: 테이블 생성
# [1] 최초 1회 실행: 원천 데이터와 진행상황 테이블을 생성합니다.
#     일반적인 재실행에서는 ner_entity_mentions를 비우지 않습니다.
# 전체 적재 및 이후 기사 단위 증분 재처리를 위한 DB 테이블
with get_connection() as connection:
    connection.execute("""
        CREATE TABLE IF NOT EXISTS ner_entity_mentions (
            press_id TEXT, file_id TEXT NOT NULL, doc_id TEXT NOT NULL,
            paragraph_id TEXT, sentence_id TEXT NOT NULL, topic TEXT,
            form TEXT, final_form TEXT, entity_name TEXT NOT NULL,
            entity_type TEXT NOT NULL, built_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
        )
    """)
    connection.execute("CREATE INDEX IF NOT EXISTS idx_entity_mentions_doc ON ner_entity_mentions (doc_id, sentence_id)")
    connection.execute("""CREATE TABLE IF NOT EXISTS ner_entity_build_progress (
        job_name TEXT PRIMARY KEY, last_doc_id TEXT, processed_docs BIGINT DEFAULT 0,
        status TEXT, updated_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP)""")

def refresh_entity_mentions(doc_ids):
    if not doc_ids:
        return
    # 검수 화면과 같은 SQL: 두 검수 테이블의 최신 완료·수정본을 문장별로 선택합니다.
    # 전달한 기사만 조회하므로 검수 테이블 전체를 매 배치 재집계하지 않습니다.
    from src.ner_review_app.modules.entity_catalog import ENTITY_MENTIONS_INSERT_SQL

    with get_connection() as connection:
        connection.execute("DELETE FROM ner_entity_mentions WHERE doc_id = ANY(%s)", (doc_ids,))
        connection.execute(ENTITY_MENTIONS_INSERT_SQL, (doc_ids,))

In [70]:
# [실행 순서 2] 저장된 진행상황 조회
# [2] 저장된 진행상황을 읽고 마지막으로 처리된 doc_id 다음부터 이어서 실행합니다.
# 최초 실행: progress 테이블이 없거나 마지막 상태가 없으면 처음부터 배치 처리
JOB_NAME = 'ner_entity_mentions_full'
BATCH_SIZE = 500
with get_connection() as connection:
    progress = connection.execute("SELECT last_doc_id, processed_docs FROM ner_entity_build_progress WHERE job_name = %s", (JOB_NAME,)).fetchone()
last_doc_id = progress[0] if progress and not hasattr(progress, 'keys') else (progress['last_doc_id'] if progress else '')
processed_docs = progress[1] if progress and not hasattr(progress, 'keys') else (progress['processed_docs'] if progress else 0)

In [ ]:
# [실행 순서 3] 전체 데이터 500건씩 배치 처리
# [3] 최초 전체 적재 시 실행합니다. 각 배치는 커밋된 후 다음 배치로 진행합니다.
#     중단된 경우 이 셀을 다시 실행하면 ner_entity_build_progress부터 이어서 처리합니다.
while True:
    with get_connection() as connection:
        batch = connection.execute("SELECT DISTINCT doc_id FROM public.ner_documents WHERE doc_id > %s ORDER BY doc_id LIMIT %s", (last_doc_id, BATCH_SIZE)).fetchall()
    batch_ids = [r[0] if not hasattr(r, 'keys') else r['doc_id'] for r in batch]
    if not batch_ids:
        break
    refresh_entity_mentions(batch_ids)
    last_doc_id, processed_docs = batch_ids[-1], processed_docs + len(batch_ids)
    with get_connection() as connection:
        connection.execute("""INSERT INTO ner_entity_build_progress (job_name, last_doc_id, processed_docs, status) VALUES (%s, %s, %s, %s) ON CONFLICT (job_name) DO UPDATE SET last_doc_id=EXCLUDED.last_doc_id, processed_docs=EXCLUDED.processed_docs, status=EXCLUDED.status, updated_at=CURRENT_TIMESTAMP""", (JOB_NAME, last_doc_id, processed_docs, 'running'))
    print(f'processed_docs={processed_docs}, last_doc_id={last_doc_id}')
# [4] 검수 내용이 수정되면 refresh_entity_mentions(['DOC_ID'])를 호출해 해당 기사를 재처리합니다.
#     문장 단위 수정도 해당 문서 전체를 재처리하면 안전하게 반영됩니다.
# [5] refresh_entity_mentions 실행 후 이 셀을 다시 실행해 최종 집계 테이블을 갱신합니다.
with get_connection() as connection:
    connection.execute("UPDATE ner_entity_build_progress SET status = 'completed', updated_at = CURRENT_TIMESTAMP WHERE job_name = %s", (JOB_NAME,))

# 특정 기사 재처리 예시: refresh_entity_mentions(['DOC_ID'])
# 특정 문장 수정 시에는 해당 문장이 포함된 doc_id만 전달하면 안전하게 재생성됨
with get_connection() as connection:
    connection.execute("DROP TABLE IF EXISTS ner_entity_frequency")
    connection.execute("DROP TABLE IF EXISTS ner_entity_location")
    connection.execute("""CREATE TABLE ner_entity_frequency AS SELECT entity_name, entity_type, COUNT(*) AS mention_count FROM ner_entity_mentions GROUP BY entity_name, entity_type""")
    connection.execute("""CREATE TABLE ner_entity_location AS SELECT press_id, entity_name, entity_type, file_id, doc_id, paragraph_id, sentence_id, COUNT(*) AS mention_count FROM ner_entity_mentions GROUP BY press_id, entity_name, entity_type, file_id, doc_id, paragraph_id, sentence_id""")

In [ ]:
# 다시 초기화해야 하는 경우 이거 실행하고 위 셀 실행 
# with get_connection() as connection:
#     connection.execute("TRUNCATE TABLE ner_entity_mentions")
#     connection.execute(
#         """
#         DELETE FROM ner_entity_build_progress
#         WHERE job_name = %s
#         """,
#         ('ner_entity_mentions_full',),
#     )